# AI Agents Are Just While Loops (That's the Scary Part)

**The smallest real AI agent — one model, three tools, one `while` loop — plus the trap it builds for itself, and the one-sentence fix that only works when you write it in the right place.**

This is the companion notebook to the [DiamantAI](https://www.youtube.com/@DiamantAI) video *"AI Agents Are Just While Loops. That's the Scary Part."* Everything the film shows — the invoice run, the spiral, the rule that gets ignored in the prompt and obeyed in the loop — runs live here, in about 80 lines of Python.

## The one idea

An AI agent is **a text file with an engine attached**. The model remembers nothing between turns; the loop re-sends the whole conversation every time, so *the transcript is the agent's entire mind* — and a mind that trusts whatever is written in it can fill up with its own mistakes. That has one practical consequence, and you'll watch it happen below: **a rule written into the prompt is advice; a rule written into the loop is physics.**

## Overview

1. Build the smallest working agent: one model, three tools, a `while` loop
2. Run it on a real question and watch the transcript grow
3. **Spring the trap** — make a tool fail politely and watch the agent spiral
4. Fix attempt #1: the rule in the **system prompt** (spoiler: ignored)
5. Fix attempt #2: the same rule **in the loop as code** (recovers on the next turn)

**Requirements:** `pip install anthropic`, and an `ANTHROPIC_API_KEY` in your environment. Each full run costs a few cents.


## 1. Setup — a folder of invoices for the agent to work on

The agent needs a real job. We'll give it the same one from the film: three invoices in a folder, and one question — *which of these is overdue, and by how much?*


In [ ]:
import os, json, subprocess
from pathlib import Path

os.environ.setdefault("AGENT_TODAY", "2026-08-30")   # the "today" the agent reasons against

Path("invoices").mkdir(exist_ok=True)
Path("invoices/invoice-august.txt").write_text(
    "INVOICE 2026-041\nClient: Meridian Systems\nAmount due: $4,200\nDue date: 2026-08-19\nStatus: unpaid\n")
Path("invoices/invoice-september.txt").write_text(
    "INVOICE 2026-052\nClient: Meridian Systems\nAmount due: $1,850\nDue date: 2026-09-15\nStatus: unpaid\n")
Path("invoices/invoice-october.txt").write_text(
    "INVOICE 2026-063\nClient: Northwind Ltd\nAmount due: $980\nDue date: 2026-10-02\nStatus: unpaid\n")
print(sorted(os.listdir("invoices")))


## 2. Three tiny tools

The model only ever writes text. The trick that turns writing into doing is a promise **your code** makes: whatever the model asks for in a structured format (a *tool call*), your program will actually run — and paste the result back into the conversation.

Note the two knobs on `run_tool`: `FLAKY` (the trap we'll spring in section 4) and the `failed_calls` guard (the fix we'll enable in section 6). Both are off for now.


In [ ]:
TOOLS = [
    {"name": "list_files", "description": "List the files in a folder.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}},
    {"name": "read_file", "description": "Read a text file and return its contents.",
     "input_schema": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}},
    {"name": "run_command", "description": "Run a shell command and return stdout+stderr.",
     "input_schema": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}},
]

FLAKY = False          # section 4 flips this: invoice reads start failing politely
GUARD = False          # section 6 flips this: the never-repeat rule, enforced by the LOOP
failed_calls = set()   # every (tool, args) pair that has already failed

def run_tool(name, args):
    key = name + json.dumps(args, sort_keys=True)
    if GUARD and key in failed_calls:
        return "BLOCKED by the loop: this exact call already failed. Change the path, the tool, or the question."
    try:
        if FLAKY and name == "read_file" and "invoice" in args.get("path", ""):
            return "ERROR: resource temporarily unavailable (errno 35), try again"
        if name == "list_files":
            return "\n".join(sorted(os.listdir(args["path"])))
        if name == "read_file":
            return Path(args["path"]).read_text()
        if name == "run_command":
            r = subprocess.run(args["command"], shell=True, capture_output=True, text=True, timeout=60)
            return (r.stdout + r.stderr).strip() or f"(exit {r.returncode}, no output)"
    except Exception as e:
        return f"ERROR: {e}"

def note_failure(name, args, out):
    if out.startswith("ERROR"):
        failed_calls.add(name + json.dumps(args, sort_keys=True))


## 3. The entire program

This is the part the film shows once, huge, because it **is** the whole machine. Send the conversation to the model. Run whatever it asks for. Paste the result back in. Go around again. Every agent framework you've heard of — LangChain, CrewAI, AutoGen — wraps this exact loop with nicer handles.

Two production defenses are already here, and both live **in the loop, not in the prompt**: the hard stop (`MAX_TURNS`) and, later, the never-repeat guard.


In [ ]:
import anthropic

MODEL = "claude-sonnet-5"
MAX_TURNS = 10
client = anthropic.Anthropic()   # uses ANTHROPIC_API_KEY from your environment

def run_agent(question, extra_system=""):
    system = f"You are an agent. Use the tools to answer. Today is {os.environ['AGENT_TODAY']}." + extra_system
    transcript = [{"role": "user", "content": question}]   # <-- the agent's entire mind

    for turn in range(1, MAX_TURNS + 1):                   # the hard stop lives in the LOOP
        reply = client.messages.create(model=MODEL, max_tokens=1024, system=system,
                                       tools=TOOLS, messages=transcript)
        transcript.append({"role": "assistant", "content": reply.content})
        calls = [b for b in reply.content if b.type == "tool_use"]
        for b in reply.content:
            if b.type == "text" and b.text.strip():
                print(f"[turn {turn}] {b.text.strip()}")
        if not calls:
            print(f"(done in {turn} turns)")
            return transcript
        results = []
        for c in calls:
            out = run_tool(c.name, c.input)
            note_failure(c.name, c.input, out)
            print(f"[turn {turn}] -> {c.name}({json.dumps(c.input)})\n{out}\n")
            results.append({"type": "tool_result", "tool_use_id": c.id, "content": out})
        transcript.append({"role": "user", "content": results})   # errors append exactly like results
    print(f"(hit the {MAX_TURNS}-turn cap; stopping)")
    return transcript


## 3b. Run it

Watch the turns: on turn 1 the model doesn't answer — it asks for the file list. Turn 2, it reads all three invoices and does the date arithmetic itself (nobody gave it a calculator). Turn 3, it answers — checked against files it had never seen.


In [ ]:
FLAKY, GUARD = False, False
failed_calls.clear()
transcript = run_agent("Which of the invoices in ./invoices is overdue, and by how much?")


## 3c. Freeze it and read the mind

Between any two of those turns the model remembered **nothing** — every turn is a brand-new call to a stranger. It only looks continuous because the loop re-sends this whole pile every time. Read it: this pile of text is everything the agent *is*. **The transcript is the mind.**


In [ ]:
for i, msg in enumerate(transcript):
    if isinstance(msg["content"], str):
        print(f"{i}. [{msg['role']}] {msg['content'][:100]}")
    else:
        for b in msg["content"]:
            t = getattr(b, "type", None) or b.get("type")
            if t == "text":         print(f"{i}. [{msg['role']}] text: {b.text[:90]}")
            elif t == "tool_use":   print(f"{i}. [{msg['role']}] tool_use: {b.name}({json.dumps(b.input)})")
            elif t == "tool_result":print(f"{i}. [{msg['role']}] tool_result: {str(b['content'])[:90]}")


## 4. Spring the trap

Now we make every invoice read fail with the friendliest error in production: *temporarily unavailable, try again*. Watch what the loop does with that. The error lands in the transcript **exactly like a real result** — because to the loop, everything is a result. The model re-reads its mind, sees *try again*, and does the reasonable thing: it trusts its own notebook.

From the film's real run (the same code you're holding):

```text
[turn 2] -> read_file(invoice-august.txt)   ERROR: temporarily unavailable, try again
[turn 3] -> read_file(invoice-august.txt)   ERROR: temporarily unavailable, try again
[turn 4] -> read_file(invoice-august.txt)   ERROR: temporarily unavailable, try again
[turn 5] -> run_command(cat invoice-august.txt)   INVOICE 2026-041 ...
```

**That's the spiral** — a mind filling with its own mistake, at full price per lap. This run escaped on turn 5 (the model gave up on the broken tool and went through the shell), but it burned three paid turns first, and a slightly nastier error would have held the trap shut for good.


In [ ]:
FLAKY, GUARD = True, False
failed_calls.clear()
_ = run_agent("Which of the invoices in ./invoices is overdue, and by how much?")


## 5. Fix attempt #1 — the rule in the system prompt

The obvious fix is one sentence: *never repeat a tool call that already failed*. Write it into the system prompt — the standing instructions at the very top of the transcript, re-read every single turn. It cannot be missed.

In the film's real run, the model read that rule — **and retried the dead call three more times anyway.** A prompt is advice. It lands in the same pile as everything else, and by then the pile also holds three errors saying *try again*. Three fresh errors are louder than one old rule.


In [ ]:
FLAKY, GUARD = True, False
failed_calls.clear()
_ = run_agent("Which of the invoices in ./invoices is overdue, and by how much?",
              extra_system=" After a failed tool call, never repeat it: change the path, the tool, or the question.")


## 6. Fix attempt #2 — the same sentence, in the loop

Now move the same rule out of the mind and into the engine: `GUARD = True` makes `run_tool` **refuse** any call that already failed. Same rule, same words. In the prompt it was a suggestion the spiral could shout over. In the loop, it's physics.

The film's real run with the guard on:

```text
[turn 3] -> read_file(invoice-august.txt)
BLOCKED by the loop: this exact call already failed. Change the path, the tool, or the question.
[turn 4] -> run_command(cat invoice-august.txt; ...)   INVOICE 2026-041 ...
```

One blocked attempt — and on the very next turn the model switched tools and the run recovered.


In [ ]:
FLAKY, GUARD = True, True
failed_calls.clear()
_ = run_agent("Which of the invoices in ./invoices is overdue, and by how much?")


## The fact to walk away with

An AI agent is a text file with an engine attached — and the file can catch fire. So anything that **must** be true belongs in the engine, written as code:

- a **spending cap** and a **hard stop** (`MAX_TURNS` — a loop that can't quit is the only true disaster),
- the **never-repeat rule** (`failed_calls` — enforced, not suggested),
- and a **curated transcript** (summarize old turns, keep the facts, drop the dead weight — a shorter, truer mind beats a complete one).

The file is for thinking. The engine is for promises.

---

*From the [DiamantAI](https://www.youtube.com/@DiamantAI) video \"AI Agents Are Just While Loops. That's the Scary Part.\" — subscribe for the next one: somewhere in those turns, your agent will hand you an answer it invented, and it won't know it did. Why it **can't** know is the next film.*
